# Spatiotemporal feature-engineering shotgun — XGBoost

This notebook screens **all registered feature families** against your existing selected-binary XGBoost baseline. The baseline already contains historical comparable-sale features. Every feature variant is fitted on the same training sample and predicted on the same unseen held-out homes.

**Default experiment:** all `SCREEN_FEATURE_VARIANTS` (including the six new advanced families), 10,000 training sales, up to 1,000 August 2025 validation sales, and 220 XGBoost trees. Residual corrections are **disabled by default**, since we already evaluated those separately. Switch `RUN_RESIDUALS=True` in the settings cell only if you want to rerun the older residual family experiments.

Results are exploratory: these sampled, reduced-tree scores should **not** match the full-history 700-tree XGBoost (7.82% MdAPE). The validation month must be before December 2025; reserve later months for confirmation, since December 2025–May 2026 has already been used repeatedly.

**Files:** `summary.csv`, `predictions.csv`, `failures.json` in the unique `OUTPUT_DIR` below. The experiment does not modify saved baseline models or the production prediction pipeline.


In [ ]:
import pandas as pd
from IPython.display import display
from spacetime_fast import run_shotgun, SCREEN_FEATURE_VARIANTS
from models.spacetime_screen_residuals import RESIDUAL_VARIANTS

VALIDATION_MONTH = "2025-08"
TRAIN_START = "2022-01-01"
TRAIN_CAP = 10000             # sample of historical completed sales
VALIDATION_CAP = 1000         # holdout sample, excluded from all training
SCREEN_TREES = 220           # small screen; full model uses 700
SEED = 42
OUTPUT_DIR = "outputs/spacetime_features_all"

RUN_RESIDUALS = False        # residual corrections tested separately
CALIBRATION_CAP = 650        # used only when RUN_RESIDUALS is True
CALIBRATION_FOLDS = 2
RESIDUAL_STRENGTH = 0.5

FEATURES = list(SCREEN_FEATURE_VARIANTS)
RESIDUALS = list(RESIDUAL_VARIANTS) if RUN_RESIDUALS else []

print(f"Feature candidates ({len(FEATURES)}):", FEATURES)
print("Residual candidates:", RESIDUALS)
print("Output:", OUTPUT_DIR)


Feature candidates (16): ['baseline', 'market', 'multiscale', 'time_adjusted_comp', 'spacetime_rbf', 'spacetime_splines', 'combined', 'adaptive_comps', 'hedonic_index', 'geo_context', 'property_spacetime', 'kalman_market', 'advanced_combined', 'comp_age_first', 'comp_type_quota', 'comp_both']
Residual candidates: []
Output: outputs/spacetime_features_all


In [3]:
scores, predictions, failures = run_shotgun(
    validation_month=VALIDATION_MONTH,
    train_start=TRAIN_START,
    train_cap=TRAIN_CAP,
    validation_cap=VALIDATION_CAP,
    calibration_cap=CALIBRATION_CAP,
    calibration_folds=CALIBRATION_FOLDS,
    screen_trees=SCREEN_TREES,
    seed=SEED,
    features=FEATURES,
    residuals=RESIDUALS,
    residual_strength=RESIDUAL_STRENGTH,
    output_dir=OUTPUT_DIR,
)
display(scores)
if failures:
    print(f"{len(failures)} failed candidates; see {OUTPUT_DIR}/failures.json")
    display(pd.DataFrame(failures)[["arm", "variant", "error"]])
else:
    print("All requested candidates finished successfully.")


Shotgun: 10,000 historical sales; 1,000 unseen 2025-08 homes; 220 trees per XGBoost
  feature  baseline              MdAPE 10.769% MAPE 13.949% (ΔMdAPE +0.000pp)
  feature  market                MdAPE 10.851% MAPE 14.058% (ΔMdAPE +0.082pp)
  feature  multiscale            MdAPE 11.397% MAPE 14.095% (ΔMdAPE +0.628pp)
  feature  time_adjusted_comp    MdAPE 11.029% MAPE 13.997% (ΔMdAPE +0.260pp)
  feature  spacetime_rbf         MdAPE 10.902% MAPE 13.938% (ΔMdAPE +0.134pp)
  feature  spacetime_splines     MdAPE 10.163% MAPE 13.453% (ΔMdAPE -0.606pp)
  feature  combined              MdAPE 10.772% MAPE 13.815% (ΔMdAPE +0.003pp)
  feature  adaptive_comps        MdAPE 11.041% MAPE 13.988% (ΔMdAPE +0.272pp)
  feature  hedonic_index         MdAPE 11.175% MAPE 13.962% (ΔMdAPE +0.407pp)
  feature  geo_context           MdAPE 10.850% MAPE 13.907% (ΔMdAPE +0.082pp)
  feature  property_spacetime    MdAPE 11.183% MAPE 13.961% (ΔMdAPE +0.415pp)
  feature  kalman_market         MdAPE 11.050% MAPE 14.091

,homes,mape,mdape,median_signed,arm,variant,delta_mdape_pp,delta_mape_pp
5,1000,13.453334,10.162843,3.204010,feature,spacetime_splines,-0.605768,-0.495698
14,1000,13.749434,10.562291,3.137263,feature,comp_type_quota,-0.206321,-0.199597
15,1000,13.744451,10.726760,3.347975,feature,comp_both,-0.041851,-0.204580
0,1000,13.949031,10.768611,4.176564,feature,baseline,0.000000,0.000000
6,1000,13.814680,10.771746,3.831843,feature,combined,0.003135,-0.134351
9,1000,13.906599,10.850362,3.940973,feature,geo_context,0.081751,-0.042432
1,1000,14.058353,10.850932,3.579570,feature,market,0.082321,0.109321
4,1000,13.937722,10.902437,3.241279,feature,spacetime_rbf,0.133825,-0.011310
3,1000,13.997365,11.028786,3.674087,feature,time_adjusted_comp,0.260175,0.048334
7,1000,13.987774,11.040946,3.586078,feature,adaptive_comps,0.272334,0.038742


All requested candidates finished successfully.


In [ ]:
# Compare feature candidates to the same reduced-sample XGBoost baseline.
display(scores.sort_values(["arm", "mdape"]))
display(scores.groupby("arm")[["mdape", "mape", "delta_mdape_pp", "delta_mape_pp"]]
        .agg(["min", "median"]).round(3))

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, max(5, 0.4 * len(scores) + 1)))
p = scores.sort_values("delta_mdape_pp")
ax.barh(p["arm"] + " / " + p["variant"], p["delta_mdape_pp"])
ax.axvline(0, linewidth=1, color="gray")
ax.set_xlabel("MdAPE change from baseline (percentage points; lower is better)")
ax.set_title("Spatiotemporal feature shotgun")
fig.tight_layout()
plt.show()


In [ ]:
# Inspect whether variants help particular property types.
import data
sales = data.load("listings")
sold_month = pd.to_datetime(sales["date_sold"]).dt.to_period("M")
names = sales.loc[sold_month == pd.Period(VALIDATION_MONTH),
                  ["ml_num", "house_type_name"]]
annotated = predictions.merge(names, on="ml_num", how="left", validate="many_to_one")
subgroups = (annotated.groupby(["arm", "variant", "house_type_name"])
             .agg(n=("ml_num", "size"), mdape=("abs_pct_error", "median"))
             .query("n >= 10")
             .sort_values(["house_type_name", "mdape"]))
display(subgroups.head(50))
print(f"Full result CSV: {OUTPUT_DIR}/predictions.csv")


## Next: refine the winners

Review `summary.csv` and `failures.json`. Keep 3–5 candidates with the strongest MdAPE/MAPE and reasonable runtime. Confirm them across multiple **earlier** validation months in `spacetime_three_month.ipynb` using 40,000 training examples and 500 trees; for that notebook set `FEATURE_VARIANTS` to the shortlist and `RESIDUAL_VARIANTS=[]`.

A small change on one month can be noise or selection bias. Only promote persistent improvements to full-history 700-tree validation. The baseline and shortlisted variants must use identical chronological splits and target definitions.

If your local repo is behind, update it and open this notebook in Jupyter:

```powershell
git switch main
git pull origin main
uv sync
uv run jupyter lab
```

For a quicker initial screen, change `TRAIN_CAP=5000`, `VALIDATION_CAP=400`, and `SCREEN_TREES=100`; for a stronger screen increase those settings, but retain the same holdout across variants.


---

## Structural spatiotemporal shotgun — GPBoost, Matérn GP, local kernels

**This is a separate experiment from the feature-engineering shotgun above.** Run the new cells below; the earlier feature-screening cells do **not** need to be rerun.

We test actual model structures, not just new features:

| Model | Role |
| --- | --- |
| `baseline` | Matched tuned **XGBoost + splines + geo** |
| `xgb_tensor_gam` | Matched XGBoost + splines + geo **+ Tensor GAM** (strong incumbent) |
| `xgb_st_kernel` / `xgb_st_kernel_type` | Out-of-time local space-time kernel corrections |
| `xgb_spatial_matern` | Exact spatial Matérn GP correction |
| `xgb_st_matern` | Exact three-dimensional space-time Matérn GP correction |
| `gpboost_vecchia` | Independent joint boosted-trees + spatiotemporal GP model (optional package) |

All methods use the **same sampled training period** and **identical held-out homes**, with strict chronological cutoffs. Corrections use *earlier out-of-time errors*, never their own training errors.

**Screening is not final validation.** Our full-history off-market best remains **7.4857% MdAPE**. The pilot trains fewer XGBoost trees and sales; compare *deltas against this pilot's matched incumbent*, **not** raw MdAPE against the full-history best. A one-month smoke test is for functional checks only. The three-month test helps select candidates for full-history confirmation.

Implementations: `models/spacetime_structural_features.py`, `models/spacetime_structural_residuals.py`, `models/spacetime_gpboost.py`. The root `spacetime_structural_shotgun.py` only coordinates evaluation, checkpointing and comparisons. GPBoost requires `%pip install gpboost` once; if absent it is logged as unavailable rather than counted as an inferior model.


In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from spacetime_structural_shotgun import run_structural_shotgun, METHODS
from models.spacetime_structural_features import SPLINE_GEO_SUFFIX

assert SPLINE_GEO_SUFFIX == "__st_spacetime_splines+geo_context"
assert "gpboost_vecchia" in METHODS
assert "xgb_tensor_gam" in METHODS

# Independent output folders prevent the small smoke run from contaminating
# a larger three-month experiment. Each run resumes only identical settings.
RUN_STRUCTURAL_SMOKE = True
SMOKE_SETTINGS = dict(
    months=("2025-10",),
    train_start="2022-01-01",
    train_cap=5_000,
    validation_cap=250,
    calibration_cap=150,
    calibration_folds=2,
    trees=250,
    gpboost_rounds=180,
    max_gp_points=250,
    residual_strength=0.5,
    methods=METHODS,
    seed=42,
    bootstrap_repeats=250,
    output_dir="outputs/spacetime_structural_notebook_smoke",
    resume=True,
)
print("Structural candidates:", ", ".join(METHODS))
print("Pilot data/trees:", SMOKE_SETTINGS["train_cap"], SMOKE_SETTINGS["trees"])


In [ ]:
# STEP 1 — one-month smoke / initial comparison.
# Run this cell before the full three-month pilot.
if RUN_STRUCTURAL_SMOKE:
    structural_smoke, structural_smoke_monthly, structural_smoke_predictions, structural_smoke_failures = (
        run_structural_shotgun(**SMOKE_SETTINGS)
    )
    display(structural_smoke[[
        "variant", "homes", "mdape", "mape",
        "delta_mdape_pp", "delta_mape_pp",
        "runtime_seconds", "triage",
    ]].round(4))

    if not structural_smoke_failures.empty:
        print("Unavailable/failed variants (NOT classified as model losses):")
        display(structural_smoke_failures[[
            "month", "variant", "error_type", "message"
        ]])
    print("Smoke outputs:", SMOKE_SETTINGS["output_dir"])


In [ ]:
# STEP 2 — inspect which models are even plausible against our incumbent.
# Negative change means an improvement. This cell uses the smoke-run results.
if RUN_STRUCTURAL_SMOKE and "structural_smoke" in globals():
    columns = [
        "variant", "delta_mdape_pp", "delta_mape_pp",
        "delta_vs_tensor_mdape_pp", "delta_vs_tensor_mape_pp",
        "runtime_seconds", "triage",
    ]
    display(structural_smoke[[
        c for c in columns if c in structural_smoke.columns
    ]].round(4))

    if "delta_vs_tensor_mdape_pp" in structural_smoke:
        plot_data = structural_smoke.sort_values("delta_vs_tensor_mdape_pp")
        fig, ax = plt.subplots(figsize=(9, max(4, 0.55 * len(plot_data))))
        ax.barh(plot_data.variant, plot_data.delta_vs_tensor_mdape_pp)
        ax.axvline(0, color="gray", linewidth=1)
        ax.invert_yaxis()
        ax.set_xlabel("ΔMdAPE vs matched spline + geo + Tensor GAM (pp; lower better)")
        ax.set_title("Structural space-time smoke test — exploratory")
        fig.tight_layout()
        plt.show()

    print("One month cannot reliably eliminate model classes.")
    print("Use the three-month pilot below before shortlisting models.")


### Three-month development pilot (opt-in)

After the smoke test succeeds, change `RUN_STRUCTURAL_THREE_MONTH` to `True` and run the cell below. This uses 8,000 past sales, 500 trees and held-out sales from September, October and November 2025. It will take more computation than the smoke test.

Prioritize methods that repeatedly improve **both MdAPE and MAPE**, have acceptable runtime, and beat the **matched Tensor GAM** rather than just the uncorrected XGBoost. The `triage` column is a conservative screening heuristic, not proof that any model can or cannot win after tuning.

Changing training budgets changes the experimental fingerprint: use a different `output_dir` instead of mixing caches. Completed monthly predictions are checkpointed and reused with `resume=True`.


In [ ]:
# STEP 3 — more discriminating screen on three earlier monthly holdouts.
RUN_STRUCTURAL_THREE_MONTH = False   # Enable after reviewing smoke output

THREE_MONTH_SETTINGS = dict(
    months=("2025-09", "2025-10", "2025-11"),
    train_start="2022-01-01",
    train_cap=8_000,
    validation_cap=900,
    calibration_cap=450,
    calibration_folds=2,
    trees=500,
    gpboost_rounds=220,
    max_gp_points=350,
    residual_strength=0.5,
    methods=METHODS,
    seed=42,
    bootstrap_repeats=500,
    output_dir="outputs/spacetime_structural_notebook_three_month",
    resume=True,
)

if RUN_STRUCTURAL_THREE_MONTH:
    structural_pooled, structural_monthly, structural_predictions, structural_failures = (
        run_structural_shotgun(**THREE_MONTH_SETTINGS)
    )
    display(structural_pooled.round(4))
    display(structural_monthly.round(4))
    if not structural_failures.empty:
        display(structural_failures[[
            "month", "variant", "error_type", "message"
        ]])
    print("Full pilot outputs:", THREE_MONTH_SETTINGS["output_dir"])


---

## GPBoost failure diagnosis — what is making its predictions go wrong?

The first October 2025 smoke test showed roughly **48.8% MdAPE** for the independent GPBoost model. **Do not interpret that as an untunable method yet.** We need to determine whether GPBoost's *tree mean*, the *Gaussian-process random effect*, or their combination is responsible.

The structural smoke XGBoost uses the *tuned* model parameters **except** for its 250 trees / 5,000 historical training sales. The three-month XGBoost uses 500 trees / 8,000 historical sales; the established full-history validation used 700 trees and an expanding historical population. Thus, compare models **within each identical validation fold**, never compare the raw 250-home pilot MdAPE to the full-history MdAPE of 7.4857%.

**First diagnostic cell:** inspect the already saved GPBoost predictions and their absolute/signed errors without fitting anything.

**Second diagnostic cell:** optionally refit GPBoost *once* with the same October smoke train/holdout and inspect its **tree-only**, **GP random-effect**, and **combined** log predictions. In Gaussian GPBoost, `response_mean = fixed_effect + random_effect_mean` on the centered log-target scale. The diagnostic checks that identity instead of assuming the prediction scale is correct. We retain the original prediction calculation; this is instrumentation, not a new model.

GPBoost's 180-round LightGBM-style tree with a different learning rate and regularization is **not** hyperparameter-matched to our highly tuned XGBoost, even if it receives the same feature family. Treat the tree-only metrics as an essential sanity check.


In [ ]:
# STEP A: This reads predictions ALREADY produced by the smoke test.
# No new model fitting, no extra GPBoost training time.
if "structural_smoke_predictions" not in globals():
    raise RuntimeError("First run the Structural spatiotemporal shotgun smoke cell above.")

smoke_audit = structural_smoke_predictions.copy()
smoke_audit["ml_num"] = smoke_audit["ml_num"].astype(str)
audit_models = ["baseline", "xgb_tensor_gam", "gpboost_vecchia"]
available = smoke_audit.loc[smoke_audit["variant"].isin(audit_models)].copy()
if "gpboost_vecchia" not in set(available["variant"]):
    print("GPBoost did not produce successful smoke predictions; inspect smoke failures.")
else:
    available["signed_error_pct"] = 100 * (
        available["predicted_price"] - available["sold_price"]
    ) / available["sold_price"]
    available["ape"] = available["signed_error_pct"].abs()
    available["price_ratio"] = (
        available["predicted_price"] / available["sold_price"]
    )
    overview = available.groupby("variant").agg(
        n=("ml_num", "size"),
        mdape=("ape", "median"),
        mape=("ape", "mean"),
        median_signed_pct=("signed_error_pct", "median"),
        median_predicted_to_actual=("price_ratio", "median"),
        predicted_price_median=("predicted_price", "median"),
        actual_price_median=("sold_price", "median"),
    )
    display(overview.round(4))
    print("GPBoost signed percentage-error quantiles:")
    display(available.loc[available.variant == "gpboost_vecchia",
                          "signed_error_pct"].quantile([.01,.10,.25,.50,.75,.90,.99]))
    print("Actual and predicted GPBoost price quantiles:")
    display(available.loc[available.variant == "gpboost_vecchia",
                          ["sold_price", "predicted_price"]]
            .quantile([.01,.10,.50,.90,.99]).round(0))
    print("Interpretation: systematic under/overprediction indicates a scale/intercept")
    print("or mean-model problem; widespread deviations indicate broader model failure.")


In [ ]:
# STEP B: Optional deep diagnosis, requires ONE NEW GPBoost fit on the
# SAME homes and training sample as the October smoke run.
# It can take substantially longer than the other residual corrections.
RUN_GPBOOST_DEEP_DIAGNOSTIC = False  # Set True to inspect the tree and GP parts

if RUN_GPBOOST_DEEP_DIAGNOSTIC:
    import importlib
    import numpy as np
    import data
    import models.spacetime_gpboost as gpboost_module
    from spacetime_fast import _cap
    gpboost_module = importlib.reload(gpboost_module)

    assert len(SMOKE_SETTINGS["months"]) == 1, (
        "The deep diagnostic uses a single smoke month."
    )
    origin = pd.Period(SMOKE_SETTINGS["months"][0], freq="M")
    sales = data.load("listings")
    sales["date_sold"] = pd.to_datetime(sales["date_sold"])
    train_start = pd.Timestamp(SMOKE_SETTINGS["train_start"])
    train = _cap(
        sales.loc[(sales.date_sold >= train_start) &
                  (sales.date_sold < origin.start_time)],
        SMOKE_SETTINGS["train_cap"], SMOKE_SETTINGS["seed"],
    )
    held = _cap(
        sales.loc[sales.date_sold.dt.to_period("M") == origin],
        SMOKE_SETTINGS["validation_cap"], SMOKE_SETTINGS["seed"] + 11,
    )
    assert len(held) and len(train)
    hidden = data.hide_answers(held, "off_market")

    # This returns separate, UNROUNDED log/price components after a fresh fit.
    details = gpboost_module.gpboost_joint(
        train, hidden, rounds=SMOKE_SETTINGS["gpboost_rounds"],
        seed=SMOKE_SETTINGS["seed"], return_components=True,
    )
    diagnostic = pd.DataFrame({
        "ml_num": held.ml_num.astype(str).to_numpy(),
        "sold_price": held.sold_price.to_numpy(dtype=float),
        "gpboost_combined": details["price_combined"],
        "gpboost_tree_only": details["price_tree_only"],
        "gp_effect_log": details["log_random_effect"],
        "fixed_effect_log": details["log_fixed"],
        "joint_log": details["log_combined"],
    })
    baseline_predictions = structural_smoke_predictions.loc[
        (structural_smoke_predictions.variant == "baseline") &
        (structural_smoke_predictions.month == str(origin)),
        ["ml_num", "predicted_price"],
    ].copy()
    baseline_predictions["ml_num"] = baseline_predictions.ml_num.astype(str)
    if baseline_predictions.ml_num.duplicated().any():
        raise ValueError("Duplicate saved XGBoost validation keys")
    if set(baseline_predictions.ml_num) != set(diagnostic.ml_num):
        raise ValueError("Cached smoke baseline is from a different holdout cohort")
    diagnostic = diagnostic.merge(
        baseline_predictions.rename(columns={
            "predicted_price": "xgboost_splines_geo"
        }), on="ml_num", validate="one_to_one",
    )

    def component_score(label, predicted):
        error = 100 * np.abs(predicted - diagnostic.sold_price) / diagnostic.sold_price
        signed = 100 * (predicted - diagnostic.sold_price) / diagnostic.sold_price
        return {
            "model": label, "mdape": error.median(), "mape": error.mean(),
            "median_signed_pct": signed.median(),
            "median_predicted_price": np.median(predicted),
        }

    results = [
        component_score(col, diagnostic[col])
        for col in ("xgboost_splines_geo", "gpboost_tree_only", "gpboost_combined")
    ]
    display(pd.DataFrame(results).sort_values("mdape").round(4))
    print("Check Gaussian decomposition (response_mean = fixed + random):")
    print("Maximum log-decomposition gap:", details["log_decomposition_gap"])
    print("Max tree-only vs fixed-effect gap:", details["log_tree_vs_fixed_gap"])
    print("Random-effect log quantiles:")
    display(diagnostic.gp_effect_log.quantile([0,.01,.1,.5,.9,.99,1]))
    print("Joint & tree-only log-prediction quantiles:")
    display(diagnostic[["fixed_effect_log", "joint_log"]]
            .quantile([.01,.1,.5,.9,.99]).round(4))
    print("Training log-target range:",
          details["train_log_target_min"], details["train_log_target_max"])
    print("GPBoost model metadata:",
          {k: details[k] for k in (
              "training_rows", "feature_count", "categorical_feature_count",
              "model_rounds", "train_target_offset"
          )})

    output_path = Path(SMOKE_SETTINGS["output_dir"]) / "gpboost_component_diagnostic.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)
    diagnostic.to_csv(output_path, index=False)
    print("Saved paired diagnostic per home to:", output_path)

    if details["log_decomposition_gap"] > 1e-4:
        print("WARNING: GPBoost response and latent decomposition disagree.")
    elif results[1]["mdape"] < results[2]["mdape"] * .7:
        print("The tree-only component looks substantially better than combined.")
        print("Investigate GP covariance or GP-effect extrapolation.")
    else:
        print("If tree-only error is also severe, investigate tree capacity,")
        print("feature encoding, centering, and GPBoost hyperparameters first.")
